# 03 — Absolute Sustainability Ratio

    ASR = emissions / allocated carrying capacity

The numerator is notebook 02's emissions table. The denominator is each
country's fair share of the global carbon budget under **equal per capita**
allocation — every person alive gets the same share.

The budget itself is the **IPCC AR6 remaining carbon budget for &lt;1.5C**
(category C1), not a fixed annual allocation: it shrinks every year as the
world keeps emitting, which is what "fair share of the carbon budget" means
in climate policy. 1.5C, specifically, because it's the threshold Pacific
nations themselves have campaigned for at every COP — *"1.5 to stay alive."*

C1 bundles 12 different AR6 model/policy runs, each implying a different
budget. We report one reference run rather than an unlabelled blend: see
`config.CC_SCENARIO` for which, and why.

- **ASR < 1** — living within a 1.5C-consistent fair share
- **ASR > 1** — exceeding it, by that multiple

**Outputs**
- `data_viz/asr.json` — `{iso3: {year: asr}}` for the D3 app
- `data_viz/asr.csv` — long form

In [5]:
import json

import pandas as pd
import pyaesa

from config import (
    ASR_FILE, CC_CATEGORY, CC_SCENARIO, FU_CODE, LCA_FILE, LCA_VERSION,
    LCIA_METHOD, PROJECT, REGION_COL, ROOT, VIZ, YEARS, YEAR_COLS,
)

pyaesa.set_workspace(top_path=str(ROOT))

Workspace setup guidance information is available in:
/Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026/data_raw/summary.log


## 1. Compute

`r_p` is pinned to the countries actually present in the emissions table.
Without it pyaesa allocates a budget to every World Bank country and then
fails on the ones with no emissions data to divide.

`figures=False` is not optional here. Under `dynamic_ar6` pyaesa renders one
chart per country per AR6 scenario per method — 14,568 PNGs, five-plus hours,
nothing in this project reads them, and the run then dies in pyaesa's own
plotting code (`KeyError: ''` in `render_asr_figures`) *after* the results
CSV is written but *before* the export below. Leaving figures on costs hours
and loses the export.

pyaesa runs the whole chain — allocation, then the AR6 budget, then the
ratio — so there is no separate step to call first. This also (re)processes
AR6 for our 2000–2023 window the first time it runs, on top of the download
notebook 01 already did. Roughly 20 minutes with figures off.

In [6]:
countries = sorted(pd.read_csv(LCA_FILE)[REGION_COL].unique())
print(f"Computing ASR for {len(countries)} countries...")

result = pyaesa.deterministic_asr(
    project_name=PROJECT,
    source="iso3",
    fu_code=FU_CODE,
    years=list(YEARS),
    lcia_method=LCIA_METHOD,
    r_p=countries,
    base_asocc_args={"include_lcia_based_allocation_methods": False},
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {"active": True, "category": CC_CATEGORY},
    },
    lca_args={"external_lca": {"active": True, "version_name": LCA_VERSION}},
    figures=False,
)

Computing ASR for 206 countries...


## 2. Inspect

The dynamic AR6 output carries one row per country per AR6 scenario, so the
first step is filtering down to `CC_SCENARIO`. There is no `cc_bound`
min/max here the way the old static budget had — this run reports one
number per country per year, without a built-in uncertainty range.

Palau and New Caledonia still sit far above their neighbours. Both are
territorial emissions divided by a small resident population — Palau hosts
several times its own population in visitors each year, New Caledonia runs
nickel smelters. Worth flagging in the visualisation rather than presenting
flat.

In [7]:
asr = pd.read_csv(ASR_FILE)
asr = asr[asr["cc_scenario"] == CC_SCENARIO]
assert len(asr) == len(countries), (
    f"expected one row per country for {CC_SCENARIO}, got {len(asr)} for {len(countries)} countries"
)

long = (
    asr.melt(
        id_vars=[REGION_COL], value_vars=YEAR_COLS,
        var_name="year", value_name="asr",
    )
    .astype({"year": int})
    .rename(columns={REGION_COL: "iso_code"})
)

snapshot = (
    long.query("year == 2020")
    .set_index("iso_code")["asr"]
    .sort_values()
)
print(f"{len(snapshot)} countries in 2020\n")
print("Lowest 10:\n", snapshot.head(10).round(3), "\n")
print("Highest 10:\n", snapshot.tail(10).round(1))

206 countries in 2020

Lowest 10:
 iso_code
NRU    0.017
MHL    0.017
TUV    0.068
FSM    0.085
BDI    0.114
YEM    0.134
RWA    0.136
SLB    0.136
KIR    0.136
AFG    0.156
Name: asr, dtype: float64 

Highest 10:
 iso_code
TTO      5.2
BRN      6.0
BHR      7.2
QAT      7.5
CUW      7.9
PLW     12.8
MCO     24.8
SMR    187.8
GRL    259.8
BMU    307.1
Name: asr, dtype: float64


## 3. Export

In [8]:
VIZ.mkdir(exist_ok=True)
long.to_csv(VIZ / "asr.csv", index=False)

export = {
    iso: dict(zip(g["year"], g["asr"].round(4)))
    for iso, g in long.groupby("iso_code")
}
(VIZ / "asr.json").write_text(json.dumps(export, indent=2))

print(f"{len(export)} countries -> data_viz/asr.json, data_viz/asr.csv")

206 countries -> data_viz/asr.json, data_viz/asr.csv
